# 2장 SOLID 기초: 견고한 파이썬 애플리케이션 구축

파이썬으로 구현하는 클린 아키텍처 - 2장 SOLID 기초: 견고한 파이썬 애플리케이션 구축 코드 예제

## 개요

이번 장에서는 클린 아키텍처의 기반이 되는 원칙들을 더 깊이 다룬다.

### 00_srp_user_pre_refactor.py

## SRP 위반: 리팩토링 전

단일 책임 원칙(SRP)은 각 소프트웨어 모듈이 변경되어야 할 이유가 하나뿐이어야 한다는 원칙이다. 언뜻 단순해 보이지만, 실제로 정의하고 구현하기는 쉽지 않다. 간단한 예제로 ﻿살펴보자.


In [1]:
# ============================================================
# SRP(단일 책임 원칙) 위반 사례 - 리팩토링 전
# User 클래스가 사용자 데이터 관리, 게시물 생성, 타임라인 조회,
# 프로필 수정 등 여러 책임을 동시에 담당하는 구조
# ============================================================


# SRP 위반: 하나의 클래스가 4가지 책임(사용자 정보, 게시물, 타임라인, 프로필)을 모두 보유
class User:
    def __init__(self, user_id: str, username: str, email: str):
        self.user_id = user_id
        self.username = username
        self.email = email
        self.posts = []  # 게시물 목록 - 사용자 데이터와 무관한 별도 책임

    # 책임 #1: 게시물 생성 기능 (별도 클래스로 분리해야 할 대상)
    def create_post(self, content: str) -> dict:
        post = {"id": len(self.posts) + 1, "content": content, "likes": 0}
        self.posts.append(post)
        return post

    # 책임 #2: 타임라인 조회 기능 (별도 서비스로 분리해야 할 대상)
    def get_timeline(self) -> list:
        # 사용자의 타임라인을 가져와 반환
        # 팔로우 중인 사용자의 게시물을 가져오고
        # 정렬하는 복잡한 로직이 필요할 수 있음
        pass

    # 책임 #3: 프로필 수정 기능 (별도 매니저로 분리해야 할 대상)
    def update_profile(self, new_username: str = None, new_email: str = None):
        if new_username:
            self.username = new_username
        if new_email:
            self.email = new_email


### 01_spr_user_post_refactor.py

## SRP(단일 책임 원칙) - 적용 후

User 클래스를 ﻿단일 책임 원칙과 엔터티 개념에 맞게 리팩터링해 보자.

In [2]:
# ============================================================
# SRP(단일 책임 원칙) 적용 후 - 리팩토링 완료
# 기존 User 클래스의 여러 책임을 각각 독립된 클래스로 분리한 구조
# 각 클래스가 하나의 변경 이유만 가지도록 설계
# ============================================================


# User 클래스: 사용자 데이터만 보관하는 순수 엔티티
# 게시물, 타임라인, 프로필 수정 로직은 모두 별도 클래스로 분리
class User:
    def __init__(self, user_id: str, username: str, email: str):
        self.user_id = user_id
        self.username = username
        self.email = email


# 게시물 관리 전담 클래스 - 게시물 생성/관리 책임만 보유
class PostManager:
    # User 객체를 매개변수로 받아 게시물 생성 (User와의 느슨한 결합)
    def create_post(self, user: User, content: str):
        post = {
            "id": self.generate_post_id(),
            "user_id": user.user_id,
            "content": content,
            "likes": 0,
        }
        # 게시물 저장 로직
        return post

    # 게시물 고유 ID 생성 담당 메서드
    def generate_post_id(self):
        # 고유한 게시물 ID 생성 로직
        pass


# 타임라인 서비스 전담 클래스 - 타임라인 조회 책임만 보유
class TimelineService:
    def get_timeline(self, user: User) -> list:
        # 사용자의 타임라인을 가져와 반환
        # 팔로우 중인 사용자들의 게시물을 가져오고 정렬하는 복잡한 로직이 필요할 수 있음
        pass


# 프로필 관리 전담 클래스 - 프로필 수정 책임만 보유
class ProfileManager:
    def update_profile(
        self, user: User, new_username: str = None, new_email: str = None
    ):
        if new_username:
            user.username = new_username
        if new_email:
            user.email = new_email
        # 프로필 업데이트를 위한 추가 로직, 이메일 인증 트리거 등


### 02_srp_unit_test.py

## SRP 적용 후 유닛 테스트

SRP를 준수하면 각 클래스를 복잡한 설정 없이 독립적으로 테스트할 수 있다.

**주의:** SRP의 핵심은 "변경 이유가 하나"이지, "수행하는 작업이 하나"가 아니다. 과도한 분리는 오히려 전체 시스템의 이해를 어렵게 만든다.

In [ ]:
# ============================================================
# SRP 적용 후의 유닛 테스트
# 단일 책임을 가진 PostManager를 독립적으로 테스트하는 예제
# SRP 덕분에 복잡한 모의 객체 없이 간결한 테스트 작성 가능
# ============================================================

import unittest

# [수정] 원본 코드: from post_manager import PostManager
# [수정] 원본 코드: from user import User
# 노트북에서는 이전 셀에서 정의한 클래스를 직접 참조합니다.


# PostManager의 게시물 생성 기능을 검증하는 테스트 클래스
class TestPostManager(unittest.TestCase):
    # 게시물 생성 시 사용자 ID, 내용, 좋아요 수, 게시물 ID의 정확성 검증
    def test_create_post(self):
        user = User("123", "testuser", "test@example.com")
        post_manager = PostManager()
        post = post_manager.create_post(user, "Hello, world!")

        self.assertEqual(post["user_id"], "123")      # 사용자 ID 일치 여부
        self.assertEqual(post["content"], "Hello, world!")  # 게시물 내용 일치 여부
        self.assertEqual(post["likes"], 0)             # 초기 좋아요 수 (0)
        self.assertIn("id", post)                      # 게시물 ID 존재 여부

### 03_ocp_shape_pre_refactor.py

## OCP(개방-폐쇄 원칙) - 위반 사례

집중적이고 유지 보수하기 쉬운 클래스를 만드는 데 단일 책임 원칙이 어떤 역할을 하는지 살펴봤다. 이제 견고한 소프트웨어 설계의 또 다른 중요한 축인 확장성으로 넘어가 보자.

In [ ]:
# ============================================================
# OCP(개방-폐쇄 원칙) 위반 사례 - 리팩토링 전
# 새 도형을 추가할 때마다 AreaCalculator의 코드를 수정해야 하는 구조
# isinstance()로 타입을 일일이 확인하는 방식은 OCP 위반의 전형적 패턴
# ============================================================


# 사각형 클래스 - 가로, 세로 속성만 보유
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height


# 원 클래스 - 반지름 속성만 보유
class Circle:
    def __init__(self, radius):
        self.radius = radius


# OCP 위반: 새 도형 추가 시 이 클래스의 calculate_area 메서드를 반드시 수정해야 함
# isinstance()를 사용한 타입 분기 - 도형이 늘어날수록 if-elif 체인이 길어지는 문제
class AreaCalculator:
    def calculate_area(self, shape):
        if isinstance(shape, Rectangle):
            return shape.width * shape.height
        elif isinstance(shape, Circle):
            return 3.14 * shape.radius**2
        else:
            raise ValueError("지원되지 않는 도형")


# 사용법
rectangle = Rectangle(5, 4)
circle = Circle(3)

calculator = AreaCalculator()
print(f"사각형 면적: {calculator.calculate_area(rectangle)}")
print(f"원 면적: {calculator.calculate_area(circle)}")


### 04_ocp_shape_post_refactor.py

## OCP(개방-폐쇄 원칙) - 적용 후

추상 클래스와 다형성을 활용하여 OCP를 적용했다. 새로운 도형을 추가할 때 기존 코드를 수정할 필요 없이 새로운 클래스만 추가하면 된다.

In [ ]:
# ============================================================
# OCP(개방-폐쇄 원칙) 적용 후 - 리팩토링 완료
# 추상 클래스(Shape)와 다형성을 활용하여 확장에 열려 있고 수정에 닫힌 구조
# 새 도형 추가 시 기존 코드를 전혀 변경하지 않아도 되는 설계
# ============================================================

import math
from abc import ABC, abstractmethod


# 모든 도형의 공통 인터페이스 역할을 하는 추상 기본 클래스
class Shape(ABC):
    @abstractmethod
    def area(self):
        """각 도형이 자신의 넓이 계산 방법을 직접 구현하도록 강제하는 추상 메서드"""
        pass


# Shape를 상속받아 자신만의 넓이 계산 로직을 구현한 사각형 클래스
class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height


# Shape를 상속받아 자신만의 넓이 계산 로직을 구현한 원 클래스
class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius**2


# OCP 준수: AreaCalculator는 Shape 추상 타입에만 의존
# 새 도형이 추가되어도 이 클래스는 수정 불필요 (확장에 열려 있고 수정에 닫힘)
class AreaCalculator:
    def calculate_area(self, shape: Shape):
        return shape.area()


# 사용법
rectangle = Rectangle(5, 4)
circle = Circle(3)

calculator = AreaCalculator()
print(f"사각형 면적: {calculator.calculate_area(rectangle)}")
print(f"원 면적: {calculator.calculate_area(circle)}")


# OCP의 장점: AreaCalculator 수정 없이 새 도형(삼각형) 추가
class Triangle(Shape):
    def __init__(self, base, height):
        self.base = base
        self.height = height

    def area(self):
        return 0.5 * self.base * self.height


triangle = Triangle(6, 4)
# 기존 calculator 인스턴스를 그대로 사용 가능 - OCP 덕분에 무수정 확장
print(f"삼각형 면적: {calculator.calculate_area(triangle)}")


### 05_isp_media_pre_refactor.py

## ISP(인터페이스 분리 원칙) - 위반 사례

**인터페이스 분리 원칙(Interface Segregation Principle):** 클라이언트는 자신이 사용하지 않는 인터페이스에 의존하도록 강제되어서는 안 된다.

하나의 거대한 인터페이스가 모든 미디어 타입의 기능을 포함하여, 구현 클래스가 사용하지 않는 메서드까지 구현해야 한다.

In [ ]:
# ============================================================
# ISP(인터페이스 분리 원칙) 위반 사례 - 리팩토링 전
# 하나의 거대한 인터페이스(MultimediaPlayer)가 모든 미디어 기능을 포함
# 구현 클래스가 자신과 무관한 메서드까지 강제로 구현해야 하는 문제
# ============================================================

from abc import ABC, abstractmethod


# ISP 위반: 재생, 정지, 가사 표시, 비디오 필터까지 모든 기능을 하나의 인터페이스에 포함
# 이 인터페이스를 구현하는 클래스는 불필요한 메서드도 반드시 구현해야 하는 부담
class MultimediaPlayer(ABC):
    @abstractmethod
    def play_media(self, file: str) -> None:
        pass

    @abstractmethod
    def stop_media(self) -> None:
        pass

    @abstractmethod
    def display_lyrics(self, file: str) -> None:
        pass

    @abstractmethod
    def apply_video_filter(self, filter: str) -> None:
        pass


# MusicPlayer: 음악 관련 기능만 필요하지만 비디오 필터까지 구현 강제
class MusicPlayer(MultimediaPlayer):
    def play_media(self, file: str) -> None:
        # 음악 재생 구현
        print(f"음악 재생 중: {file}")

    def stop_media(self) -> None:
        # 음악 중지 구현
        print("음악 중지 중")

    def display_lyrics(self, file: str) -> None:
        # 가사 표시 구현
        print(f"{file}의 가사 표시 중")

    # ISP 위반의 전형적 증상: 사용하지 않는 메서드를 억지로 구현
    # NotImplementedError를 던지는 것 자체가 설계 결함의 신호
    def apply_video_filter(self, filter: str) -> None:
        # MusicPlayer에 대해 이 메서드는 의미가 없음
        raise NotImplementedError(
            "MusicPlayer가 지원하지 않는 비디오 필터")


class VideoPlayer(MultimediaPlayer):
    # 비디오 플레이어 구현
    ...


### 06_isp_media_post_refactor.py

## ISP(인터페이스 분리 원칙) - 적용 후

인터페이스를 작은 단위로 분리하여 각 클라이언트가 필요한 인터페이스만 구현하도록 했다. 불필요한 의존성이 제거되어 코드가 더 유연해진다.

In [ ]:
# ============================================================
# ISP(인터페이스 분리 원칙) 적용 후 - 리팩토링 완료
# 하나의 거대한 인터페이스를 역할별로 분리하여 각 클래스가
# 필요한 인터페이스만 선택적으로 구현하는 구조
# ============================================================

from abc import ABC, abstractmethod


# 미디어 재생/정지 기능만 담당하는 인터페이스 (모든 플레이어 공통)
class MediaPlayable(ABC):
    @abstractmethod
    def play_media(self, file: str) -> None:
        pass

    @abstractmethod
    def stop_media(self) -> None:
        pass


# 가사 표시 기능만 담당하는 인터페이스 (음악 플레이어 전용)
class LyricsDisplayable(ABC):
    @abstractmethod
    def display_lyrics(self, file: str) -> None:
        pass


# 비디오 필터 기능만 담당하는 인터페이스 (비디오 플레이어 전용)
class VideoFilterable(ABC):
    @abstractmethod
    def apply_video_filter(self, filter: str) -> None:
        pass


# MusicPlayer: 재생 + 가사 표시 인터페이스만 구현 (비디오 필터 불필요)
class MusicPlayer(MediaPlayable, LyricsDisplayable):
    def play_media(self, file: str) -> None:
        print(f"음악 재생 중: {file}")

    def stop_media(self) -> None:
        print("음악 중지 중")

    def display_lyrics(self, file: str) -> None:
        print(f"가사 표시 중: {file}")


# VideoPlayer: 재생 + 비디오 필터 인터페이스만 구현 (가사 표시 불필요)
class VideoPlayer(MediaPlayable, VideoFilterable):
    def play_media(self, file: str) -> None:
        print(f"동영상 재생 중: {file}")

    def stop_media(self) -> None:
        print("동영상 중지 중")

    def apply_video_filter(self, filter: str) -> None:
        print(f"비디오 필터 적용: {filter}")


# BasicAudioPlayer: 재생 기능만 필요하므로 MediaPlayable만 구현
# ISP 덕분에 불필요한 가사 표시나 비디오 필터를 구현할 필요 없음
class BasicAudioPlayer(MediaPlayable):
    def play_media(self, file: str) -> None:
        print(f"오디오 재생 중: {file}")

    def stop_media(self) -> None:
        print("오디오 중지 중")


### 07_lsp_vehicle_pre_refactor.py

## LSP(리스코프 치환 원칙) - 위반 사례

다양한 차량 타입과 연료 소비를 관리하는 시스템을 생각해 보자.

In [ ]:
# ============================================================
# LSP(리스코프 치환 원칙) 위반 사례 - 리팩토링 전
# 부모 클래스(Vehicle)를 자식 클래스(ElectricCar)로 대체했을 때
# 예상과 다른 동작이 발생하는 문제
# LSP: 자식 클래스는 부모 클래스를 대체해도 프로그램이 올바르게 동작해야 한다는 원칙
# ============================================================


# 내연기관 차량 기준으로 설계된 기본 클래스
class Vehicle:
    def __init__(self, fuel_capacity: float):
        self._fuel_capacity = fuel_capacity
        self._fuel_level = fuel_capacity  # 초기 연료량 = 최대 용량

    def fuel_level(self) -> float:
        return self._fuel_level

    def consume_fuel(self, distance: float) -> None:
        fuel_consumed = distance / 10  # 단순화를 위해 1리터당 10km 가정
        if self._fuel_level - fuel_consumed < 0:
            raise ValueError("거리를 이동하기에 충분한 연료가 없습니다")
        self._fuel_level -= fuel_consumed


# LSP 위반: ElectricCar가 Vehicle을 상속하지만 연료 소비 방식이 근본적으로 다름
# 전기차는 "연료"가 아닌 "전력"을 사용하므로 부모 클래스의 가정과 충돌
class ElectricCar(Vehicle):
    def __init__(self, battery_capacity: float):
        super().__init__(battery_capacity)

    # 부모의 consume_fuel을 오버라이드하지만, 소비율이 다름 (1kWh당 5km vs 1리터당 10km)
    # 결과적으로 drive_vehicle 함수에서 "리터" 단위로 출력하지만 실제로는 "kWh" 값
    def consume_fuel(self, distance: float) -> None:
        energy_consumed = distance / 5  # 단순화를 위해 1kWh당 5km 가정
        if self._fuel_level - energy_consumed < 0:
            raise ValueError("해당 거리를 주행할 만큼의 전력이 부족합니다")
        self._fuel_level -= energy_consumed


# 이 함수는 Vehicle 타입을 기대하며 "리터" 단위를 가정
def drive_vehicle(vehicle: Vehicle, distance: float) -> None:
    initial_fuel = vehicle.fuel_level()
    vehicle.consume_fuel(distance)
    fuel_consumed = initial_fuel - vehicle.fuel_level()
    print(f"연료 소모량: {fuel_consumed:.2f} 리터")


# 사용법
car = Vehicle(50)  # 50리터 탱크
drive_vehicle(car, 100)  # 정상 작동

electric_car = ElectricCar(50)  # 50kWh 배터리
drive_vehicle(electric_car, 100)  # 이 코드는 잘못된 연료 소비량을 출력


### 08_lsp_vehicle_post_refactor.py

## LSP(리스코프 치환 원칙) - 적용 후

리스코프 치환 원칙을 준수하도록 이것을 리팩토링해 보자. 먼저 동력원에 대한 추상 기본 클래스를 정의하는 것부터 시작한다.

In [ ]:
# ============================================================
# LSP(리스코프 치환 원칙) 적용 후 - 리팩토링 완료
# 상속 대신 구성(Composition)을 활용하여 동력원을 추상화한 구조
# 어떤 동력원이든 Vehicle에 주입하면 동일한 인터페이스로 동작
# ============================================================

from abc import ABC, abstractmethod


# 모든 동력원의 공통 인터페이스 역할을 하는 추상 기본 클래스
# 연료 탱크, 배터리 등 다양한 동력원이 이 계약을 따르는 구조
class PowerSource(ABC):
    def __init__(self, capacity: float):
        self._capacity = capacity
        self._level = capacity  # 초기 에너지 수준 = 최대 용량

    def level(self) -> float:
        return self._level

    @abstractmethod
    def consume(self, distance: float) -> float:
        """주행 거리에 따른 에너지 소비 후 소비량을 반환하는 추상 메서드"""
        pass


# 내연기관 연료 탱크 구현 - PowerSource를 상속한 구체 클래스
class FuelTank(PowerSource):
    def consume(self, distance: float) -> float:
        fuel_consumed = distance / 10  # 단순화를 위해 1리터당 10km 가정
        if self._level - fuel_consumed < 0:
            raise ValueError("해당 거리를 주행할 만큼의 연료가 부족합니다")
        self._level -= fuel_consumed
        return fuel_consumed


# 전기차 배터리 구현 - PowerSource를 상속한 구체 클래스
class Battery(PowerSource):
    def consume(self, distance: float) -> float:
        energy_consumed = distance / 5  # 단순화를 위해 1kWh당 5km 가정
        if self._level - energy_consumed < 0:
            raise ValueError("해당 거리를 주행할 만큼의 충전량이 되어 있지 않습니다")
        self._level -= energy_consumed
        return energy_consumed


# Vehicle: 동력원을 외부에서 주입받는 구성(Composition) 패턴 적용
# 어떤 PowerSource든 교체 가능하므로 LSP를 자연스럽게 준수
class Vehicle:
    def __init__(self, power_source: PowerSource):
        self._power_source = power_source  # 동력원을 생성자 주입으로 받음

    def power_level(self) -> float:
        return self._power_source.level()

    def drive(self, distance: float) -> float:
        return self._power_source.consume(distance)


# 통일된 인터페이스로 모든 차량 타입을 동일하게 처리하는 함수
def drive_vehicle(vehicle: Vehicle, distance: float) -> None:
    try:
        energy_consumed = vehicle.drive(distance)
        print(f"소비된 에너지: {energy_consumed:.2f} 단위")  # 단위가 통일되어 혼동 없음
    except ValueError as e:
        print(f"여행 완료 불가: {e}")


# 사용법 - 동력원만 교체하면 차량 타입이 자연스럽게 변경
fuel_car = Vehicle(FuelTank(50))  # 50리터 탱크
drive_vehicle(fuel_car, 100)  # 출력: 소비된 에너지: 10.00 단위

electric_car = Vehicle(Battery(50))  # 50kWh 배터리
drive_vehicle(electric_car, 100)  # 출력: 소비된 에너지: 20.00 단위


### 09_dip_a_b_dependency.py

## DIP(의존성 역전 원칙) - 기본 개념

**의존성 역전 원칙(Dependency Inversion Principle):** 고수준 모듈은 저수준 모듈에 의존해서는 안 되며, 둘 다 추상화에 의존해야 한다.

이 코드는 DIP의 기본적인 의존성 관계를 보여준다.

In [ ]:
# ============================================================
# DIP(의존성 역전 원칙) - 직접 의존성의 기본 예제
# 고수준 모듈(A)이 저수준 모듈(B)을 직접 생성하여 강하게 결합된 구조
# 추상화 없이 구체 클래스에 직접 의존하는 DIP 위반의 최소 예제
# ============================================================


# 고수준 모듈: B의 구체 클래스를 직접 생성 (강한 결합)
# B를 다른 구현으로 교체하려면 A의 코드를 반드시 수정해야 하는 문제
class A:
    def __init__(self):
        self.b = B()  # 구체 클래스 B에 대한 직접 의존


# 저수준 모듈: A에 의해 직접 참조되는 구체 클래스
class B:
    def __init__(self):
        pass


### 10_dip_user_entity_pre_refactor.py

## DIP(의존성 역전 원칙) - 위반 사례

이 간단한 예제가 의존성 역전 원칙이 풀려는 문제를 이해하는 출발점이다. 많은 소프트웨어 시스템에서 고수준 모듈(핵심 비즈니스 로직)이 저수준 모듈(구체적인 구현 ﻿세부 사항)에 의존하곤 한다.

In [ ]:
# ============================================================
# DIP(의존성 역전 원칙) 위반 사례 - 리팩토링 전
# 비즈니스 로직(UserEntity)이 특정 데이터베이스(MySQL)에 직접 의존
# 데이터베이스를 변경하면 비즈니스 로직도 함께 수정해야 하는 문제
# ============================================================


# DIP 위반: 고수준 모듈(비즈니스 로직)이 저수준 모듈(MySQL)을 직접 생성
# MySQL을 PostgreSQL 등으로 바꾸려면 이 클래스의 코드를 수정해야 하는 구조
class UserEntity:
    def __init__(self, user_id: str):
        self.user_id = user_id
        self.database = MySQLDatabase()  # 저수준 모듈에 대한 직접 의존성

    # 특정 데이터베이스 구현에 강하게 결합된 저장 메서드
    def save(self):
        self.database.insert("users", {"id": self.user_id})


# 저수준 모듈: 특정 데이터베이스의 구체적 구현
class MySQLDatabase:
    def insert(self, table: str, data: dict):
        print(f"MySQL의 {table} 테이블에 {data} 삽입 중")


### 11_dip_user_entity_post_refactor.py

## DIP(의존성 역전 원칙) - 적용 후

추상 인터페이스(Repository)를 도입하여 의존성 방향을 역전시켰다. 비즈니스 로직은 추상화에만 의존하고, 구체적인 데이터베이스 구현은 외부에서 주입된다. 이것이 클린 아키텍처의 핵심 원리이다.

In [ ]:
# ============================================================
# DIP(의존성 역전 원칙) 적용 후 - 리팩토링 완료
# 추상 인터페이스(DatabaseInterface)를 도입하여 의존성 방향을 역전
# 비즈니스 로직은 추상화에만 의존하고, 구체 구현은 외부에서 주입
# 이것이 클린 아키텍처의 핵심 원리
# ============================================================

from abc import ABC, abstractmethod


# 데이터베이스 연산의 추상 인터페이스 (고수준 모듈과 저수준 모듈 사이의 계약)
# 모든 데이터베이스 구현이 이 인터페이스를 따르도록 강제
class DatabaseInterface(ABC):
    @abstractmethod
    def insert(self, table: str, data: dict):
        pass


# DIP 적용: 구체 클래스가 아닌 추상 인터페이스(DatabaseInterface)에만 의존
# 생성자를 통해 외부에서 데이터베이스 구현을 주입받는 구조 (의존성 주입)
class UserEntity:
    def __init__(self, user_id: str, database: DatabaseInterface):
        self.user_id = user_id
        self.database = database  # 추상 타입으로 선언 - 어떤 DB 구현이든 주입 가능

    def save(self):
        self.database.insert("users", {"id": self.user_id})


# DatabaseInterface를 구현한 MySQL 전용 클래스
class MySQLDatabase(DatabaseInterface):
    def insert(self, table: str, data: dict):
        print(f"MySQL의 {table} 테이블에 {data} 삽입 중")


# DatabaseInterface를 구현한 PostgreSQL 전용 클래스
# 새 데이터베이스 추가 시 UserEntity 코드 수정 불필요 (OCP도 함께 준수)
class PostgreSQLDatabase(DatabaseInterface):
    def insert(self, table: str, data: dict):
        print(f"PostgreSQL의 {table} 테이블에 {data} 삽입 중")


# 사용법 - 원하는 데이터베이스 구현을 자유롭게 주입
mysql_db = MySQLDatabase()
user = UserEntity("123", mysql_db)
user.save()
postgres_db = PostgreSQLDatabase()
another_user = UserEntity("456", postgres_db)
another_user.save()


# 테스트용 모의(Mock) 데이터베이스 - DIP 덕분에 테스트 용이성 확보
# 실제 DB 없이도 비즈니스 로직을 검증할 수 있는 가짜 구현
class MockDatabase(DatabaseInterface):
    def __init__(self):
        self.inserted_data = []  # 삽입된 데이터를 메모리에 기록하는 리스트

    def insert(self, table: str, data: dict):
        self.inserted_data.append((table, data))


# 테스트에서 - MockDatabase를 주입하여 DB 연결 없이 동작 검증
mock_db = MockDatabase()
user = UserEntity("test_user", mock_db)
user.save()
assert mock_db.inserted_data == [("users", {"id": "test_user"})]
